# 06. Обучение на расширенном временном ряде 2001-2020

**Цель:** обучить модель на 20-летнем ряду вместо 5-летнего. Это даёт:
- 15 тренировочных пар вместо 1 (sliding windows)
- Возможность temporal held-out тестирования
- Защита от обвинений в переобучении на 2018-2022

**Архитектура та же** (ConvLSTM v2), меняется только DataLoader.

**Pre-requisites:**
1. Запущен `scripts/gee_export_2000_2020.py` и файлы скачаны с Drive в `data/extended_export/`
2. Запущен `scripts/rasterize_to_tensor_v2.py` для сборки в `tensor_01deg_extended.npz`
3. Целевой TTOP пересчитан для всех лет: `y_new_extended.npz`

**Acceptance:** RMSE на test (2021-2023) сопоставим с baseline (≤ +0.3°C).

In [ ]:
# Environment auto-detection
# Работает на Colab VM, локальном Jupyter, и Colab UI с local runtime
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    # Local Jupyter (standalone ИЛИ Colab UI + local runtime)
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}. Установи AI4ARCTIC_HOME env var или скопируй проект в ~/Desktop/Ai4Arctic"

import sys
sys.path.insert(0, str(BASE_DIR))

## 1. Загружаем extended тензор + target

Если ещё не собирал — сначала прогони:
1. `scripts/gee_export_2000_2020.py` (на 2-3 часа)
2. Скачай файлы с Drive в `data/extended_export/`
3. `scripts/rasterize_to_tensor_v2.py` адаптированный под новые годы

In [ ]:
from src.data_extended import (
    load_extended_tensor,
    make_sliding_windows,
    temporal_split,
    ExtendedTensorDataset,
)

EXTENDED_TENSOR = DATA_DIR / 'tensor_01deg_extended.npz'
EXTENDED_TARGET = DATA_DIR / 'y_new_extended.npz'

if not EXTENDED_TENSOR.exists():
    raise FileNotFoundError(
        f"Нет {EXTENDED_TENSOR}. Сначала прогони scripts/gee_export_2000_2020.py "
        f"и собери тензор через scripts/rasterize_to_tensor_v2.py."
    )

tensor, lats, lons, years = load_extended_tensor(EXTENDED_TENSOR)
target = np.load(EXTENDED_TARGET)['y_new']

print(f"Tensor: {tensor.shape}")
print(f"Target: {target.shape}")
print(f"Годы: {years}")
print(f"  диапазон: {years[0]}-{years[-1]}, всего {len(years)} лет")

## 2. Создаём sliding windows и temporal split

In [ ]:
WINDOW_SIZE = 5

windows = make_sliding_windows(years, window_size=WINDOW_SIZE)
print(f"Создано окон: {len(windows)}")
for w in windows[:3]:
    print(f"  input {w.input_years} -> target {w.target_year}")
if len(windows) > 3:
    print(f"  ... ещё {len(windows) - 3} окон")

splits = temporal_split(windows, train_until=2018, val_until=2020)
print(f"\nTrain: {len(splits['train'])} окон, target_years: {sorted([w.target_year for w in splits['train']])}")
print(f"Val:   {len(splits['val'])} окон, target_years: {sorted([w.target_year for w in splits['val']])}")
print(f"Test:  {len(splits['test'])} окон, target_years: {sorted([w.target_year for w in splits['test']])}")

## 3. Готовим DataLoader

Для экономии GPU — режем на patches 64×64.

In [ ]:
PATCH_SIZE = 64
PATCHES_PER_WINDOW = 16  # 16 patches на одно окно = 16 * 14 train okon = 224 train samples
BATCH_SIZE = 4

train_ds = ExtendedTensorDataset(tensor, target, splits['train'],
                                  patch_size=PATCH_SIZE,
                                  patches_per_window=PATCHES_PER_WINDOW)
val_ds = ExtendedTensorDataset(tensor, target, splits['val'],
                                patch_size=PATCH_SIZE,
                                patches_per_window=PATCHES_PER_WINDOW)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds)} samples, val: {len(val_ds)} samples")
print(f"Batches: train={len(train_loader)}, val={len(val_loader)}")

# Sanity check
X_sample, y_sample = train_ds[0]
print(f"\nSample X shape: {X_sample.shape}, dtype: {X_sample.dtype}")
print(f"Sample y shape: {y_sample.shape}, dtype: {y_sample.dtype}")

## 4. Обучение

In [ ]:
from src.model import ConvLSTMNet

in_channels = tensor.shape[1]
model = ConvLSTMNet(in_channels=in_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

EPOCHS = 80
EARLY_STOP_PATIENCE = 15

best_val_rmse = float('inf')
best_epoch = 0
patience = 0
history = {'train_rmse': [], 'val_rmse': []}

print(f"Обучение на расширенном временном ряду: {EPOCHS} эпох, early stop {EARLY_STOP_PATIENCE}")
for epoch in range(EPOCHS):
    # train
    model.train()
    train_losses = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred.squeeze(), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
    
    # validate
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            p = model(xb).squeeze().cpu().numpy()
            val_preds.append(p)
            val_targets.append(yb.numpy())
    val_preds = np.concatenate([p.reshape(-1) for p in val_preds])
    val_targets = np.concatenate([t.reshape(-1) for t in val_targets])
    
    train_rmse = np.sqrt(np.mean(train_losses))
    val_rmse = np.sqrt(np.mean((val_preds - val_targets) ** 2))
    history['train_rmse'].append(train_rmse)
    history['val_rmse'].append(val_rmse)
    
    if val_rmse < best_val_rmse - 1e-3:
        best_val_rmse = val_rmse
        best_epoch = epoch
        patience = 0
        torch.save({
            'state_dict': model.state_dict(),
            'epoch': epoch,
            'val_rmse': val_rmse,
            'in_channels': in_channels,
        }, MODELS_DIR / 'convlstm_ttop_rk_v2_extended.pt')
    else:
        patience += 1
        if patience >= EARLY_STOP_PATIENCE:
            print(f"Early stop at epoch {epoch}, best epoch was {best_epoch} (val_rmse={best_val_rmse:.4f})")
            break
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}: train_rmse={train_rmse:.4f}, val_rmse={val_rmse:.4f}")

## 5. Кривая обучения

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_rmse'], label='train')
ax.plot(history['val_rmse'], label='val')
ax.axvline(best_epoch, color='r', ls='--', alpha=0.5, label=f'best epoch={best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('RMSE')
ax.set_title('Обучение на extended timeseries 2001-2020')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_curve_extended.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test на 2021-2023

Это true held-out: модель никогда не видела эти годы во время обучения.

In [ ]:
# Восстанавливаем best checkpoint
ckpt = torch.load(MODELS_DIR / 'convlstm_ttop_rk_v2_extended.pt', map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()

# Test loader (без патчирования — даём полные карты)
if len(splits['test']) > 0:
    test_ds = ExtendedTensorDataset(tensor, target, splits['test'], patch_size=None)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    rows = []
    for i, (xb, yb) in enumerate(test_loader):
        target_year = splits['test'][i].target_year
        xb = xb.to(device)
        with torch.no_grad():
            pred = model(xb).squeeze().cpu().numpy()
        target_arr = yb.squeeze().numpy()
        mask = ~(np.isnan(pred) | np.isnan(target_arr))
        err = pred[mask] - target_arr[mask]
        rmse = float(np.sqrt(np.mean(err ** 2)))
        bias = float(np.mean(err))
        rows.append({
            'target_year': target_year,
            'rmse': rmse,
            'bias': bias,
            'n_valid_pixels': int(mask.sum()),
        })
        print(f"  Test target={target_year}: RMSE={rmse:.3f}, bias={bias:+.3f}")
    
    test_df = pd.DataFrame(rows)
    test_df.to_csv(METRICS_DIR / 'extended_temporal_test.csv', index=False)
    print(f"\nTest avg RMSE: {test_df['rmse'].mean():.3f}")
else:
    print("Нет test окон (target > 2020). Проверь years в тензоре.")

## 7. Сравнение с baseline

Baseline v2 модель обучена только на 2018-2022. Если extended даёт сопоставимый RMSE
— значит мы получили больше данных + temporal robustness без потери качества.

In [ ]:
print(f"Baseline v2 (5 лет training):")
print(f"  best val RMSE: 0.74 °C (из memory проекта)")
print(f"  best epoch: 20")
print()
print(f"Extended v3 (20 лет training, sliding windows):")
print(f"  best val RMSE: {best_val_rmse:.3f} °C")
print(f"  best epoch: {best_epoch}")
if len(splits['test']) > 0:
    print(f"  test RMSE (true held-out 2021-2023): {test_df['rmse'].mean():.3f} °C")

## 8. Что добавить в Главу 4 отчёта

```
Для повышения статистической мощности обучения и оценки временной обобщающей
способности модели тренировочный временной ряд расширен с 5 лет (2018-2022) до
20 лет (2001-2020). Использован метод sliding windows с окном 5 лет: вход —
5 предыдущих лет признаков, выход — MAGT в целевом году. Это даёт 15 пар
(вместо 1) и позволяет провести temporal held-out тестирование.

Архитектура модели не изменена; модифицирован только DataLoader. Разбиение:
train — target_year ∈ [2006, 2018] (13 пар), val — [2019, 2020] (2 пары),
test — [2021, 2023] (3 пары, true held-out — модель не видела эти годы).

Результат: best val RMSE = X.XX°C, test RMSE = Y.YY°C, что [сопоставимо/лучше]
с baseline (0.74°C). Сравнимая или улучшенная производительность на временно
изолированном тесте — аргумент против обвинений в data leakage.
```